The general workflow of this is:
- get artists song data (and subsequently their cowrite data).
- generate "basic" features on that data that are not very network oriented
- run a model with the basic features predict artist success
- generate features that capture nepotistic behavior in the network
- run a model with the basic features and the new network-based features
- compare models

In [ ]:
import json

with open("data/top_1000_artist_data_full.json", "r") as f:
    data = json.load(f)

print(len(data))       # number of items

940


In [41]:
import pandas as pd

In [42]:
[row for row in data if row["artist_name"]and "Taylor" in row['artist_name']][0]['works']

[{'id': 185230,
  'name': 'Pour Some Sugar on Me',
  'release_date': '1987-01-01',
  'genres': [],
  'collaborators': [{'mbid': '7bc12c8f-8022-4942-af94-a83c7cd8af55',
    'name': 'Joe Elliott',
    'roles': ['writer']},
   {'mbid': '1999f41e-058f-4221-8c4b-d7379b00cceb',
    'name': 'Phil Collen',
    'roles': ['writer']},
   {'mbid': 'aa85e56c-856d-4381-91f2-074c426d93a8',
    'name': 'Rick Savage',
    'roles': ['writer']},
   {'mbid': '61f2ae93-ab59-4c59-aef6-43f5f9afceae',
    'name': 'Robert John “Mutt” Lange',
    'roles': ['writer']},
   {'mbid': 'bf9249e1-cc5d-4154-866e-afb99c6a09cf',
    'name': 'Steve Clark',
    'roles': ['writer']}]},
 {'id': 265876,
  'name': 'Photograph',
  'release_date': '1983-01-01',
  'genres': ['hard rock'],
  'collaborators': [{'mbid': '7bc12c8f-8022-4942-af94-a83c7cd8af55',
    'name': 'Joe Elliott',
    'roles': ['writer']},
   {'mbid': '544eb865-b376-43ed-991b-863044ee5267',
    'name': 'Pete Willis',
    'roles': ['writer']},
   {'mbid': 'aa85e

In [43]:
from datetime import datetime

def parse_year_safe(date_str):
    if not date_str or not isinstance(date_str, str):
        return None
    y = date_str.split("-")[0]
    if not (y.isdigit() and len(y) == 4):
        return None
    year = int(y)
    current_year = datetime.now().year
    # ignore implausible or future years
    if year < 1950 or year > current_year:
        return None
    return year

def extract_features(artist, n_songs=5):
    works = artist.get("works", [])
    if not works:
        return None

    # Use only the first n_songs, if specified
    if n_songs is not None:
        works = works[:n_songs]

    num_works = len(works)
    if num_works == 0:
        return None

    artist_mbid = artist.get("mbid")
    collab_counts = []
    unique_collabs = set()
    valid_years = []

    for w in works:
        if not isinstance(w, dict):
            continue

        # Collaborators (excluding the artist themself)
        collabs = [c for c in w.get("collaborators", []) if c.get("mbid") != artist_mbid]
        collab_counts.append(len(collabs))
        for c in collabs:
            if c.get("mbid"):
                unique_collabs.add(c["mbid"])

        # Release year
        y = parse_year_safe(w.get("release_date"))
        if y is not None:
            valid_years.append(y)

    avg_collaborators = sum(collab_counts) / num_works if num_works else 0.0
    avg_diversity = len(unique_collabs) / num_works if num_works else 0.0
    return {
        "artist_mbid": artist_mbid,
        "artist_name": artist.get("artist_name"),
        "avg_collaborators_per_song": avg_collaborators,
        "avg_diversity": avg_diversity,
    }


# Process all artists
records = [extract_features(a) for a in data if extract_features(a) is not None]
df = pd.DataFrame(records)

df.shape

(836, 4)

we cut the artists songs off at the first 5, since we're trying to predict early artist success. That is, using the metrics of the first 5 songs, we're predicting the artists current success.

In [44]:
df.head()

,artist_mbid,artist_name,avg_collaborators_per_song,avg_diversity
0,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd,2.2,2.2
1,a74b1b7f-71a5-4011-9441-d0b5e4122711,Radiohead,5.2,2.2
2,f6beac20-5dfe-4d1f-ae02-0b0a740aafd6,"Tyler, The Creator",0.2,0.2
3,20244d07-534f-4eff-b4d4-930878889970,Taylor Swift,4.0,1.8
4,381086ea-f511-4aba-bdf9-71c753dc5077,Kendrick Lamar,3.8,3.8


In [45]:
def print_artist_years(data, artist_name="Taylor Swift"):
    for artist in data:
        if artist.get("artist_name", "").lower() == artist_name.lower():
            print(f"Years for {artist['artist_name']}:")
            years = []
            for w in artist.get("works", []):
                y = parse_year_safe(w.get("release_date"))
                if y:
                    years.append(y)
            years = sorted(set(years))
            print(years)
            print(f"Total valid works: {len(artist['works'])}")
            return
    print(f"No artist found with name '{artist_name}'")

# Example usage
print_artist_years(data, "Taylor Swift")


Years for Taylor Swift:
[1953, 1964, 1969, 1976, 1977, 1978, 1981, 1983, 1984, 1987, 1989, 1992, 1995, 1996, 1998, 2000, 2001, 2002, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Total valid works: 343


In [46]:
def find_bad_dates(data, artist_name="Taylor Swift"):
    if not isinstance(data, list):
        print("Data should be a list of artist dictionaries.")
        return

    for artist in data:
        # Ensure this entry is a dict and has an artist name
        if not isinstance(artist, dict):
            continue
        name = artist.get("artist_name")
        if not isinstance(name, str):
            continue  # skip invalid or missing artist_name
        
        # Compare case-insensitively
        if name.lower() == artist_name.lower():
            for w in artist.get("works", []) or []:
                release_date = w.get("release_date") if isinstance(w, dict) else None
                if isinstance(release_date, str) and release_date.startswith("1928"):
                    print("Bad date found:", w.get("name"), release_date)

find_bad_dates(data)

Bad date found: Silent Night 1928-01-01


In [47]:
import pandas as pd

expanded_artist_metrics = pd.read_csv('data/expanded_artist_metrics.csv')
expanded_artist_metrics.head()

,mbid,listeners,playcount
0,777a21a8-0d0f-4cf3-86b4-65bc0eba5649,2121606,49349863
1,703c557f-82bb-4646-ae2f-b5a3ef3f6148,0,0
2,25848dee-8562-4a78-b375-3a80b61da629,19286,135707
3,3e7bcc53-53d4-41d8-afbc-69f82b858fd9,31,403
4,43a0ce3a-a3d5-45c0-b944-bd8f5e021237,1,1


In [48]:


# Merge on mbid columns
merged = df.merge(
    expanded_artist_metrics,
    how="left",               # keep all artists from your features df
    left_on="artist_mbid",
    right_on="mbid"
)

# Drop the duplicate mbid column if you like
merged = merged.drop(columns=["mbid"])

merged.head()

,artist_mbid,artist_name,avg_collaborators_per_song,avg_diversity,listeners,playcount
0,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd,2.2,2.2,5052236.0,1.031561e+09
1,a74b1b7f-71a5-4011-9441-d0b5e4122711,Radiohead,5.2,2.2,7939386.0,1.277161e+09
2,f6beac20-5dfe-4d1f-ae02-0b0a740aafd6,"Tyler, The Creator",0.2,0.2,4094636.0,9.712308e+08
3,20244d07-534f-4eff-b4d4-930878889970,Taylor Swift,4.0,1.8,5682776.0,3.439117e+09
4,381086ea-f511-4aba-bdf9-71c753dc5077,Kendrick Lamar,3.8,3.8,4855042.0,9.616991e+08


In [49]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

cols_to_scale = ["avg_collaborators_per_song", "avg_diversity", "listeners",'playcount']

merged[cols_to_scale] = scaler.fit_transform(merged[cols_to_scale])

In [50]:
merged.to_csv("artist_features_ready.csv", index=False)
df = merged

In [51]:
df.head()

,artist_mbid,artist_name,avg_collaborators_per_song,avg_diversity,listeners,playcount
0,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd,0.200000,0.200000,0.571669,0.299917
1,a74b1b7f-71a5-4011-9441-d0b5e4122711,Radiohead,0.472727,0.200000,0.899301,0.371334
2,f6beac20-5dfe-4d1f-ae02-0b0a740aafd6,"Tyler, The Creator",0.018182,0.018182,0.463001,0.282373
3,20244d07-534f-4eff-b4d4-930878889970,Taylor Swift,0.363636,0.163636,0.643222,1.000000
4,381086ea-f511-4aba-bdf9-71c753dc5077,Kendrick Lamar,0.345455,0.345455,0.549291,0.279602


First basic model with just average collaborators and diversity. Realistically this could have many more "basic" features, such as top genre(s), metrics improvement over first five songs, spotify "danceability", genre diversity, etc.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor

# --- inputs ---
feature_cols = [
    "avg_collaborators_per_song",
    "avg_diversity",
]
target_col = "listeners"      # <-- or "lastfm_playcount"

# Basic hygiene
use = df.dropna(subset=feature_cols + [target_col]).copy()

X = use[feature_cols].values
y = use[target_col].values.reshape(-1, 1)

# Pipeline: scale features, MLP with early stopping

base = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=1000,
        early_stopping=True,
        n_iter_no_change=20,
        random_state=42
    ))
])

# Wrap with log1p/expm1 transform on the target
reg = TransformedTargetRegressor(
    regressor=base,
    func=np.log1p,    # y -> log(1+y) for training
    inverse_func=np.expm1
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y.ravel(), test_size=0.2, random_state=42
)

reg.fit(X_train, y_train)
pred = reg.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("R^2:", r2_score(y_test, pred))


MAE: 0.12680762172832954
R^2: -0.0631036655006143


This performs very poorly (negative R^2 suggests that the model explains literally none of the variance in the data) for a variety of reasons. 1 is that the basic features aren't descriptive enough. Its possible that more features would help (potentially more granular ones such as average general lyric semantics), but it seems like network features are necessary. 

Note that this model was a really basic attempt at predicting the data, there are many things that could be done to improve it, including just better configuration.

this loads in the node features, which we extracted with a separate r script

In [57]:
node_features = pd.read_csv('graphs/initial_graph/node_features.csv')
node_features.head()

,mbid,clustering_local,triangles,open_wedges,gwesp_contrib,degree,eigencentrality,pagerank,kcore,betweenness,closeness,same_genre_share
0,777a21a8-0d0f-4cf3-86b4-65bc0eba5649,1.0,1,0,0.0,2,0.000025,0.000181,2,0.000000,0.149267,1.000000
1,703c557f-82bb-4646-ae2f-b5a3ef3f6148,1.0,66,0,0.0,12,0.000147,0.000413,12,0.000000,0.170750,0.916667
2,25848dee-8562-4a78-b375-3a80b61da629,1.0,10,0,0.0,5,0.000000,0.000241,5,0.000000,0.430380,1.000000
3,3e7bcc53-53d4-41d8-afbc-69f82b858fd9,1.0,15,0,0.0,6,0.005511,0.000285,6,0.000000,0.230261,0.500000
4,43a0ce3a-a3d5-45c0-b944-bd8f5e021237,0.4,4,6,0.0,5,0.000005,0.000396,3,0.002118,0.147093,0.400000


In [ ]:
node_features = node_features.drop_duplicates(subset=['mbid'])

In [60]:
node_features.shape


(2806, 12)

In [61]:
df.head()

,artist_mbid,artist_name,avg_collaborators_per_song,avg_diversity,listeners,playcount
0,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd,0.200000,0.200000,0.571669,0.299917
1,a74b1b7f-71a5-4011-9441-d0b5e4122711,Radiohead,0.472727,0.200000,0.899301,0.371334
2,f6beac20-5dfe-4d1f-ae02-0b0a740aafd6,"Tyler, The Creator",0.018182,0.018182,0.463001,0.282373
3,20244d07-534f-4eff-b4d4-930878889970,Taylor Swift,0.363636,0.163636,0.643222,1.000000
4,381086ea-f511-4aba-bdf9-71c753dc5077,Kendrick Lamar,0.345455,0.345455,0.549291,0.279602


In [64]:
node_merged = df.merge(node_features, left_on='artist_mbid', right_on='mbid', how='left')
node_merged = node_merged.drop(columns='mbid')
print(node_merged.shape)
node_merged.head()

(836, 17)


,artist_mbid,artist_name,avg_collaborators_per_song,avg_diversity,listeners,playcount,clustering_local,triangles,open_wedges,gwesp_contrib,degree,eigencentrality,pagerank,kcore,betweenness,closeness,same_genre_share
0,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd,0.200000,0.200000,0.571669,0.299917,0.374269,64.0,107.0,0.0,19.0,3.572148e-02,0.000788,10.0,0.004090,0.243877,0.421053
1,a74b1b7f-71a5-4011-9441-d0b5e4122711,Radiohead,0.472727,0.200000,0.899301,0.371334,0.000000,0.0,0.0,0.0,0.0,5.255640e-19,0.000058,0.0,0.000000,0.000000,0.000000
2,f6beac20-5dfe-4d1f-ae02-0b0a740aafd6,"Tyler, The Creator",0.018182,0.018182,0.463001,0.282373,1.000000,21.0,0.0,0.0,7.0,1.167557e-05,0.000343,7.0,0.000000,0.144083,1.000000
3,20244d07-534f-4eff-b4d4-930878889970,Taylor Swift,0.363636,0.163636,0.643222,1.000000,1.000000,3.0,0.0,0.0,3.0,3.366370e-17,0.000404,3.0,0.000000,0.800000,1.000000
4,381086ea-f511-4aba-bdf9-71c753dc5077,Kendrick Lamar,0.345455,0.345455,0.549291,0.279602,0.275000,33.0,87.0,0.0,16.0,8.815397e-04,0.000855,6.0,0.003915,0.210066,0.562500


In [65]:
node_merged.head()

,artist_mbid,artist_name,avg_collaborators_per_song,avg_diversity,listeners,playcount,clustering_local,triangles,open_wedges,gwesp_contrib,degree,eigencentrality,pagerank,kcore,betweenness,closeness,same_genre_share
0,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd,0.200000,0.200000,0.571669,0.299917,0.374269,64.0,107.0,0.0,19.0,3.572148e-02,0.000788,10.0,0.004090,0.243877,0.421053
1,a74b1b7f-71a5-4011-9441-d0b5e4122711,Radiohead,0.472727,0.200000,0.899301,0.371334,0.000000,0.0,0.0,0.0,0.0,5.255640e-19,0.000058,0.0,0.000000,0.000000,0.000000
2,f6beac20-5dfe-4d1f-ae02-0b0a740aafd6,"Tyler, The Creator",0.018182,0.018182,0.463001,0.282373,1.000000,21.0,0.0,0.0,7.0,1.167557e-05,0.000343,7.0,0.000000,0.144083,1.000000
3,20244d07-534f-4eff-b4d4-930878889970,Taylor Swift,0.363636,0.163636,0.643222,1.000000,1.000000,3.0,0.0,0.0,3.0,3.366370e-17,0.000404,3.0,0.000000,0.800000,1.000000
4,381086ea-f511-4aba-bdf9-71c753dc5077,Kendrick Lamar,0.345455,0.345455,0.549291,0.279602,0.275000,33.0,87.0,0.0,16.0,8.815397e-04,0.000855,6.0,0.003915,0.210066,0.562500


In [67]:
scaler = MinMaxScaler()

cols_to_scale = node_merged.columns.tolist()
cols_to_scale.pop(0)
cols_to_scale.pop(0)

cols_to_scale

['avg_collaborators_per_song',
 'avg_diversity',
 'listeners',
 'playcount',
 'clustering_local',
 'triangles',
 'open_wedges',
 'gwesp_contrib',
 'degree',
 'eigencentrality',
 'pagerank',
 'kcore',
 'betweenness',
 'closeness',
 'same_genre_share']

In [70]:
node_merged[cols_to_scale] = scaler.fit_transform(node_merged[cols_to_scale])

In [71]:
node_merged.head()

,artist_mbid,artist_name,avg_collaborators_per_song,avg_diversity,listeners,playcount,clustering_local,triangles,open_wedges,gwesp_contrib,degree,eigencentrality,pagerank,kcore,betweenness,closeness,same_genre_share
0,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,The Weeknd,0.200000,0.200000,0.571669,0.299917,0.374269,0.230216,0.074669,0.0,0.322034,3.698710e-02,0.327665,0.50,0.124819,0.243877,0.421053
1,a74b1b7f-71a5-4011-9441-d0b5e4122711,Radiohead,0.472727,0.200000,0.899301,0.371334,0.000000,0.000000,0.000000,0.0,0.000000,5.441848e-19,0.000000,0.00,0.000000,0.000000,0.000000
2,f6beac20-5dfe-4d1f-ae02-0b0a740aafd6,"Tyler, The Creator",0.018182,0.018182,0.463001,0.282373,1.000000,0.075540,0.000000,0.0,0.118644,1.208924e-05,0.127675,0.35,0.000000,0.144083,1.000000
3,20244d07-534f-4eff-b4d4-930878889970,Taylor Swift,0.363636,0.163636,0.643222,1.000000,1.000000,0.010791,0.000000,0.0,0.050847,3.485641e-17,0.155228,0.15,0.000000,0.800000,1.000000
4,381086ea-f511-4aba-bdf9-71c753dc5077,Kendrick Lamar,0.345455,0.345455,0.549291,0.279602,0.275000,0.118705,0.060712,0.0,0.271186,9.127727e-04,0.357695,0.30,0.119489,0.210066,0.562500


running a new model now with all the features

In [73]:
# --- inputs ---
target_col = "listeners"      # <-- or "lastfm_playcount"

# Basic hygiene
use = node_merged.dropna(subset=cols_to_scale + [target_col]).copy()

X = use[cols_to_scale].values
y = use[target_col].values.reshape(-1, 1)

# Pipeline: scale features, MLP with early stopping
base = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=1000,
        early_stopping=True,
        n_iter_no_change=20,
        random_state=42
    ))
])

# Wrap with log1p/expm1 transform on the target
reg = TransformedTargetRegressor(
    regressor=base,
    func=np.log1p,    # y -> log(1+y) for training
    inverse_func=np.expm1
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y.ravel(), test_size=0.2, random_state=42
)

reg.fit(X_train, y_train)
pred = reg.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("R^2:", r2_score(y_test, pred))


MAE: 0.05720083883575816
R^2: 0.7256129856637501


Even a very simplistic attempt at adding network features for just the top five songs of an artist, improves the model immensely. A few things to note here:
- both the models could be improved significantly by better selection of features
    - the base model need more features that are descriptive of the data
    - the network model may need less features, potentially removing noise
- the models would both benefit from running a series of hyperparameter tests and other configuration changes.
    - We also think there are other models that could provide insight into why it predicts what is does (random forest)
- Finally, these models do their best at explaining early artist success, but are likely skewed by the fact that we only train on the top 1k artists. As we're able to train on more diverse artists with varying degrees of success, we'll be able to get a better idea of how accurate this type of model is. 